In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from brain_image.utils import setup_logging


setup_logging()

In [3]:
import logging
import pandas as pd
import json
import yaml
from pathlib import Path


def get_single_file(dir: Path, pattern: str) -> Path | None:
    paths = list(dir.rglob(pattern))

    num_results = len(paths)
    if num_results == 0:
        return None

    if num_results > 1:
        raise ValueError(f"Expected to find one results matching pattern {pattern} in dir {dir} - Found {num_results}: {tuple(paths)}")

    path = paths[0]
    return path
        

def gather_metrics(experiment_dir: Path, selected_hparams: list[str] = [], metrics_file_pattern: str = "*test/test_metrics.json") -> pd.DataFrame:
    all_metrics = []

    for exp_dir in experiment_dir.iterdir():
        metrics_path = get_single_file(exp_dir, metrics_file_pattern)
        if metrics_path is None:
            logging.warning(f"Could not find any paths in dir {exp_dir} matching pattern {'*test/test_metrics.json'}")
            continue

        logging.info(f"Loading metrics from {metrics_path}")

        with open(metrics_path, "r") as f:
            metrics = json.load(f)

        if len(selected_hparams) > 0:
            hparams_path = get_single_file(exp_dir, "*hparams.yaml")
            if hparams_path is None:
                logging.warning(f"Could not find hparam file")
                continue
            
            with open(hparams_path) as f:
                hparams = yaml.safe_load(f)

            for hparam_key in selected_hparams:
                hparam_parts = hparam_key.split(".")
                curr_hparam = hparams
                for part in hparam_parts:
                    curr_hparam = curr_hparam[part]

                metrics[hparam_key] = curr_hparam

        all_metrics.append(metrics)

    metrics = pd.DataFrame.from_records(all_metrics)
    return metrics




In [ ]:
ex_path = Path("experiments/encoders-full")
metrics = gather_metrics(ex_path, ["config.align_img_encoder", "config.eeg_encoder", "dataset_config.subs"])
metrics = metrics.rename(columns={"dataset_config.subs": "sub"})
metrics["sub"] = metrics["sub"].apply(lambda s: s[0])
#metrics.to_csv(ex_path / "experiment-metrics.csv")

metrics

13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031151853lybfym-slurmarr14533684_61/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031154502chsucz-slurmarr14533684_79/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031135117mxrisc-slurmarr14533684_2/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031154435wzeznf-slurmarr14533684_78/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031154401vrblfm-slurmarr14533684_77/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031151649pczfvf-slurmarr14533684_57/version_0/test/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/encoders-full/251031141840pvxlof-slurmarr14533684_20/version_0/test/test_metrics.json
13:29:11 | WAR

,align/brain_acc,align/image_acc,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,config.align_img_encoder,config.eeg_encoder,sub
0,0.230,0.415,0.099416,0.184108,0.434774,0.445854,0.510126,0.486859,0.983664,0.755877,aligned_synclr_vitb16,nice,2
1,0.500,0.645,0.107604,0.198295,0.466608,0.499447,0.492437,0.489573,0.979751,0.741456,aligned_synclr_vitb16,atms,10
2,0.150,0.220,0.104659,0.183206,0.468266,0.522286,0.493216,0.489121,0.983626,0.755843,clip_vitl14,nice,3
3,0.345,0.465,0.108348,0.181189,0.485226,0.497814,0.480251,0.489422,0.982953,0.753278,aligned_synclr_vitb16,atms,9
4,0.470,0.565,0.098534,0.195060,0.467513,0.465603,0.487462,0.493995,0.988411,0.756678,aligned_synclr_vitb16,atms,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.230,0.390,0.104078,0.185123,0.495427,0.524246,0.493442,0.500126,0.986018,0.747456,aligned_synclr_vitb16,nice,5
76,0.295,0.360,0.101822,0.194004,0.498970,0.514623,0.486231,0.494648,0.984175,0.748292,clip_vith14,atms,3
77,0.270,0.300,0.100442,0.184726,0.450503,0.471935,0.488065,0.511457,0.982958,0.751883,unaligned_synclr_vitb16,nice,9
78,0.090,0.140,0.107604,0.187496,0.507085,0.509573,0.499673,0.491382,0.981796,0.747370,clip_vitl14,nice,5


In [5]:
grouped_metrics = metrics.drop(columns=["sub"]).groupby(["config.align_img_encoder", "config.eeg_encoder"])
metrics_mean = grouped_metrics.mean()
metrics_std = grouped_metrics.std()
metrics_mean

align/brain_acc  align/image_acc  \
config.align_img_encoder config.eeg_encoder                                     
aligned_synclr_vitb16    atms                         0.3860           0.5035   
                         nice                         0.2910           0.4765   
clip_vith14              atms                         0.2680           0.3305   
                         nice                         0.2095           0.3185   
clip_vitl14              atms                         0.2105           0.2615   
                         nice                         0.1575           0.2205   
unaligned_synclr_vitb16  atms                         0.3310           0.4245   
                         nice                         0.2660           0.3320   

                                             prior/pixcorr  prior/ssim  \
config.align_img_encoder config.eeg_encoder                              
aligned_synclr_vitb16    atms                     0.106149    0.190163   
                         nice                     0.103233    0.188678   
clip_vith14              atms                     0.105604    0.192721   
                         nice                     0.104041    0.190765   
clip_vitl14              atms                     0.106264    0.192346   
                         nice                     0.105513    0.186065   
unaligned_synclr_vitb16  atms                     0.106521    0.193708   
                         nice                     0.102560    0.187022   

                                             prior/alex2  prior/alex5  \
config.align_img_encoder config.eeg_encoder                             
aligned_synclr_vitb16    atms                   0.470133     0.486902   
                         nice                   0.478578     0.497043   
clip_vith14              atms                   0.475540     0.489085   
                         nice                   0.483362     0.488332   
clip_vitl14              atms                   0.473570     0.489274   
                         nice                   0.488420     0.502698   
unaligned_synclr_vitb16  atms                   0.486779     0.501033   
                         nice                   0.478085     0.498201   

                                             prior/inceptionv3  prior/clip  \
config.align_img_encoder config.eeg_encoder                                  
aligned_synclr_vitb16    atms                         0.491219    0.494314   
                         nice                         0.492337    0.492565   
clip_vith14              atms                         0.491073    0.494739   
                         nice                         0.492090    0.495756   
clip_vitl14              atms                         0.482211    0.499990   
                         nice                         0.493681    0.498234   
unaligned_synclr_vitb16  atms                         0.487143    0.498555   
                         nice                         0.493234    0.500603   

                                             prior/efficientnet  prior/swav  
config.align_img_encoder config.eeg_encoder                                  
aligned_synclr_vitb16    atms                          0.984433    0.751041  
                         nice                          0.984432    0.749957  
clip_vith14              atms                          0.984524    0.748489  
                         nice                          0.984533    0.748143  
clip_vitl14              atms                          0.984247    0.749072  
                         nice                          0.983409    0.751352  
unaligned_synclr_vitb16  atms                          0.984755    0.747128  
                         nice                          0.985100    0.749584

In [6]:
metrics_std

align/brain_acc  align/image_acc  \
config.align_img_encoder config.eeg_encoder                                     
aligned_synclr_vitb16    atms                       0.073590         0.082497   
                         nice                       0.052377         0.071143   
clip_vith14              atms                       0.067132         0.070807   
                         nice                       0.061303         0.075168   
clip_vitl14              atms                       0.053668         0.056816   
                         nice                       0.047973         0.068168   
unaligned_synclr_vitb16  atms                       0.067979         0.069460   
                         nice                       0.063368         0.075506   

                                             prior/pixcorr  prior/ssim  \
config.align_img_encoder config.eeg_encoder                              
aligned_synclr_vitb16    atms                     0.005042    0.005983   
                         nice                     0.003352    0.004327   
clip_vith14              atms                     0.004870    0.003251   
                         nice                     0.005678    0.006513   
clip_vitl14              atms                     0.005345    0.005495   
                         nice                     0.003864    0.006495   
unaligned_synclr_vitb16  atms                     0.005984    0.004690   
                         nice                     0.004148    0.003492   

                                             prior/alex2  prior/alex5  \
config.align_img_encoder config.eeg_encoder                             
aligned_synclr_vitb16    atms                   0.010631     0.013299   
                         nice                   0.018923     0.026526   
clip_vith14              atms                   0.015986     0.021177   
                         nice                   0.017760     0.018176   
clip_vitl14              atms                   0.021589     0.021237   
                         nice                   0.026723     0.022859   
unaligned_synclr_vitb16  atms                   0.017728     0.018593   
                         nice                   0.017556     0.020284   

                                             prior/inceptionv3  prior/clip  \
config.align_img_encoder config.eeg_encoder                                  
aligned_synclr_vitb16    atms                         0.016298    0.008477   
                         nice                         0.018738    0.010348   
clip_vith14              atms                         0.018110    0.006205   
                         nice                         0.013544    0.008324   
clip_vitl14              atms                         0.013583    0.007418   
                         nice                         0.022556    0.012042   
unaligned_synclr_vitb16  atms                         0.020908    0.011253   
                         nice                         0.018673    0.012748   

                                             prior/efficientnet  prior/swav  
config.align_img_encoder config.eeg_encoder                                  
aligned_synclr_vitb16    atms                          0.002314    0.004980  
                         nice                          0.002023    0.004242  
clip_vith14              atms                          0.001983    0.002964  
                         nice                          0.003004    0.003036  
clip_vitl14              atms                          0.002309    0.003617  
                         nice                          0.001297    0.004625  
unaligned_synclr_vitb16  atms                          0.002585    0.003201  
                         nice                          0.002409    0.002788

In [7]:
def combine_column(mean_table, std_table, metric):
    mean_values = mean_table[metric].values
    std_values = std_table[metric].values

    formatted_values = [f"{mean_value}±{std_value}" for (mean_value, std_value) in zip(mean_values, std_values)]
    return formatted_values

combine_column(metrics_mean, metrics_std, "align/brain_acc")
    

['0.3859999909996986±0.07359046281032201',
 '0.2909999921917915±0.05237684026093557',
 '0.26799999326467516±0.0671317068495013',
 '0.2094999946653843±0.06130298278476104',
 '0.21049999296665192±0.053668219160346885',
 '0.15749999582767488±0.04797279319470647',
 '0.330999992787838±0.06797875232396901',
 '0.26599999219179155±0.06336840616614095']

In [8]:
ex_path = Path("experiments/prior_second")
metrics = gather_metrics(ex_path, ["config.prior_align_second_mode"], metrics_file_pattern="test_metrics.json")
metrics

13:29:11 | INFO     | Loading metrics from experiments/prior_second/251111230700tovphb-slurm14700182/version_0/test_metrics.json


13:29:11 | INFO     | Loading metrics from experiments/prior_second/251111230720iokhhq-slurm14700184/version_0/test_metrics.json
13:29:11 | INFO     | Loading metrics from experiments/prior_second/251111230700bahqib-slurm14700183/version_0/test_metrics.json


,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,prior/pred2_align_top1,prior/pred2_cos,config.prior_align_second_mode
0,0.155428,0.309385,0.763216,0.808367,0.655402,0.729497,0.918219,0.612828,0.155,0.800377,condition
1,0.092074,0.304061,0.670930,0.754673,0.631985,0.745452,0.933517,0.613288,NaN,NaN,none
2,0.077788,0.311172,0.535352,0.539196,0.560905,0.557990,0.958551,0.660263,0.015,0.719032,concat


In [9]:
ex_path = Path("experiments/modes")
metrics = gather_metrics(ex_path, ["config.prior_align_second_mode", "config.do_align"], metrics_file_pattern="test_metrics.json")
metrics

13:30:29 | INFO     | Loading metrics from experiments/modes/251117162427dhqcfb-slurmarr14739522_2/version_0/test_metrics.json
13:30:29 | INFO     | Loading metrics from experiments/modes/251117161924dkgqyq-slurmarr14739522_1/version_0/test_metrics.json
13:30:29 | INFO     | Loading metrics from experiments/modes/251117161751ldlpxa-slurmarr14739522_0/version_0/test_metrics.json
13:30:29 | INFO     | Loading metrics from experiments/modes/251117162524klynzm-slurmarr14739522_3/version_0/test_metrics.json


,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,prior/pred2_align_top1,prior/pred2_cos,config.prior_align_second_mode,config.do_align,align/brain_acc,align/image_acc
0,0.125272,0.323001,0.740653,0.788492,0.658417,0.746080,0.924025,0.612343,0.040,0.614638,condition,False,NaN,NaN
1,0.153480,0.310207,0.816834,0.876884,0.758668,0.825879,0.868427,0.552285,NaN,NaN,none,True,0.335,0.415
2,0.125776,0.314033,0.741482,0.779975,0.669623,0.720126,0.911747,0.602232,0.055,0.616872,condition,True,0.295,0.370
3,0.123326,0.303889,0.770276,0.833342,0.708970,0.808995,0.890512,0.576800,NaN,NaN,none,False,NaN,NaN


In [10]:
ex_path = Path("experiments/variance")
metrics = gather_metrics(ex_path, ["config.seed"], metrics_file_pattern="test_metrics.json")
metrics

13:32:33 | INFO     | Loading metrics from experiments/variance/251117161151gwwvbi-slurmarr14739501_0/version_0/test_metrics.json
13:32:33 | INFO     | Loading metrics from experiments/variance/251117161242nimlqa-slurmarr14739501_1/version_0/test_metrics.json
13:32:33 | INFO     | Loading metrics from experiments/variance/251117161648crftrb-slurmarr14739501_2/version_0/test_metrics.json
13:32:33 | INFO     | Loading metrics from experiments/variance/251117161648vggfon-slurmarr14739501_3/version_0/test_metrics.json
13:32:33 | INFO     | Loading metrics from experiments/variance/251117161649ztwtfj-slurmarr14739501_4/version_0/test_metrics.json


,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,config.seed
0,0.136876,0.311764,0.754648,0.846583,0.738241,0.821508,0.886903,0.561633,0
1,0.107365,0.313499,0.717387,0.811055,0.704548,0.791784,0.911647,0.585494,1
2,0.107994,0.307780,0.726734,0.782538,0.678844,0.760854,0.919625,0.604372,2
3,0.109347,0.303399,0.658291,0.753015,0.649899,0.739573,0.922310,0.617712,3
4,0.112422,0.308730,0.729573,0.796332,0.715854,0.773241,0.906275,0.589515,4
